In [7]:
!pip install tensorflow tensorflow-datasets matplotlib seaborn scikit-learn Pillow tqdm opencv-python-headless -q
print("✅ All packages installed")

✅ All packages installed


In [8]:
import os
import json
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from datetime import datetime
from tqdm import tqdm

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, TensorBoard
)
from tensorflow.keras.preprocessing.image import load_img, img_to_array

from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)
from PIL import Image
import cv2

warnings.filterwarnings('ignore')

# ── Reproducibility ─────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── GPU check ───────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
print(f"TensorFlow version : {tf.__version__}")
print(f"GPU available      : {'✅ YES — ' + gpus[0].name if gpus else '❌ No GPU — training will be slower'}")
print(f"Notebook started   : {datetime.now().strftime('%Y-%m-%d %H:%M')}")

TensorFlow version : 2.21.0
GPU available      : ❌ No GPU — training will be slower
Notebook started   : 2026-03-20 13:47


In [9]:
#  GLOBAL CONFIGURATION — edit these to match your setup


CONFIG = {
    # Paths
    'data_dir'    : 'data',               # Root of train/ val/ folders
    'model_dir'   : 'models',
    'output_dir'  : 'outputs',

    # Image settings
    'img_size'    : (224, 224),            # EfficientNetB0 native size
    'channels'    : 3,

    # Training hyperparameters
    'batch_size'  : 32,
    'epochs_head' : 10,                    # Epochs with frozen base
    'epochs_fine' : 20,                    # Fine-tuning epochs
    'lr_head'     : 1e-3,
    'lr_fine'     : 1e-5,
    'dropout'     : 0.4,
    'val_split'   : 0.2,

    # EfficientNet fine-tune — unfreeze top N layers
    'unfreeze_layers': 30,

    # Confidence threshold for prediction
    'confidence_threshold': 0.65,
}

# Create directories
for d in [CONFIG['model_dir'], CONFIG['output_dir'], CONFIG['data_dir']]:
    Path(d).mkdir(parents=True, exist_ok=True)

IMG_SHAPE = CONFIG['img_size'] + (CONFIG['channels'],)
print("Configuration loaded ✅")
for k, v in CONFIG.items():
    print(f"  {k:<22}: {v}")

Configuration loaded ✅
  data_dir              : data
  model_dir             : models
  output_dir            : outputs
  img_size              : (224, 224)
  channels              : 3
  batch_size            : 32
  epochs_head           : 10
  epochs_fine           : 20
  lr_head               : 0.001
  lr_fine               : 1e-05
  dropout               : 0.4
  val_split             : 0.2
  unfreeze_layers       : 30
  confidence_threshold  : 0.65


In [10]:
# Option A: Download from Kaggle 
# Paste your kaggle.json credentials when prompted, then run this cell.

USE_KAGGLE = False   # Set True if running on Colab with Kaggle API

if USE_KAGGLE:
    from google.colab import files
    uploaded = files.upload()  # Upload kaggle.json
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d emmarex/plantdisease -p data --unzip -q
    print("✅ PlantVillage dataset downloaded")
else:
    print("ℹ️  Skipped Kaggle download. Using local data/ folder.")
    print("   Download manually: https://www.kaggle.com/datasets/emmarex/plantdisease")

ℹ️  Skipped Kaggle download. Using local data/ folder.
   Download manually: https://www.kaggle.com/datasets/emmarex/plantdisease
